# 04 CNN Model

This notebook builds and evaluates a 1D CNN for software effort prediction using raw dataset loading, label encoding, StandardScaler, and a 70/15/15 train/validation/test split.

In [1]:
from importlib import import_module, util
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

drive = None
if util.find_spec("google.colab") is not None:
    drive = import_module("google.colab.drive")
    IN_COLAB = True
else:
    IN_COLAB = False


def resolve_project_root() -> Path:
    if IN_COLAB:
        drive.mount("/content/drive", force_remount=False)
        drive_root = Path("/content/drive/MyDrive")
        candidate_roots = [
            drive_root / "Software-cost-Estimation",
            drive_root / "Colab Notebooks" / "Software-cost-Estimation",
        ]
        for candidate in candidate_roots:
            if (candidate / "src").exists():
                return candidate

        for src_dir in drive_root.rglob("src"):
            if src_dir.is_dir() and src_dir.parent.name == "Software-cost-Estimation":
                return src_dir.parent

        raise FileNotFoundError(
            "Could not find the 'Software-cost-Estimation' project folder in Google Drive. "
            "Upload the full project folder to MyDrive or Colab Notebooks."
        )

    root_dir = Path.cwd()
    if not (root_dir / "src").exists() and (root_dir.parent / "src").exists():
        root_dir = root_dir.parent
    return root_dir


root_dir = resolve_project_root()
if not (root_dir / "src").exists():
    raise FileNotFoundError(f"Could not find project root containing 'src' directory. Checked: {root_dir}")

sys.path.insert(0, str(root_dir))
print("Using project root:", root_dir)

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.data_loader import load_all_raw_datasets
from src.evaluate import evaluate_predictions, save_metrics
from src.feature_engineering import inverse_log_transform, log_transform_target
from src.preprocess import preprocess_dataset

processed_dir = root_dir / "data" / "processed"
metrics_dir = root_dir / "results" / "metrics"

raw_datasets = load_all_raw_datasets()
dataset_key = "desharnais"  # Change to "cocomo81" or "china" to switch datasets.
dataset = raw_datasets[dataset_key]

target_col = dataset.columns[-1]
features_df, target_series, _ = preprocess_dataset(dataset, target_col=target_col)
X = features_df.to_numpy(dtype=np.float32)
y = target_series.to_numpy(dtype=np.float32)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=SEED,
)
validation_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=validation_ratio,
    random_state=SEED,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

# Keep the final test split untouched until the very end of evaluation.
y_train_fit = log_transform_target(y_train, use_log_transform=True).astype(np.float32)
y_val_fit = log_transform_target(y_val, use_log_transform=True).astype(np.float32)

X_train_cnn = reshape_for_cnn(X_train_scaled)
X_val_cnn = reshape_for_cnn(X_val_scaled)
X_test_cnn = reshape_for_cnn(X_test_scaled)

print("Using dataset:", dataset_key)
print("Train/val/test shapes:", X_train_cnn.shape, X_val_cnn.shape, X_test_cnn.shape)

Using project root: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation
Using dataset: desharnais
Train/val/test shapes: (56, 12, 1) (12, 12, 1) (13, 12, 1)


In [2]:
model = build_cnn_regressor(
    input_length=X_train_cnn.shape[1],
    filters=32,
    kernel_size=3,
    dense_units=64,
    learning_rate=1e-3,
)

model, history = train_cnn_model(
    model,
    X_train_cnn, y_train_fit,
    X_val_cnn, y_val_fit,
    epochs=100,
    batch_size=32,
    verbose=0,
)

print("Best validation MAE:", min(history["val_mean_absolute_error"]))

Best validation MAE: 0.20336870849132538


In [3]:
cnn_preds_log = model.predict(X_test_cnn, verbose=0).ravel()
cnn_preds = inverse_log_transform(cnn_preds_log, use_log_transform=True)
cnn_metrics = evaluate_predictions("cnn_baseline", y_test, cnn_preds)
cnn_metrics

,model,mae,rmse,r2,mape,mdmre,pred25
0,cnn_baseline,0.541122,0.759069,-0.035909,32.492124,0.276006,38.461538


In [4]:
cnn_metrics_path = save_metrics(
    cnn_metrics,
    file_name="cnn_metrics.csv",
    output_dir=metrics_dir,
 )
print("Saved CNN metrics to:", cnn_metrics_path)

Saved CNN metrics to: c:\Users\kalpi\OneDrive\Desktop\Software-cost-Estimation\results\metrics\cnn_metrics.csv


In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

from src.cnn_model import build_cnn_regressor, reshape_for_cnn, train_cnn_model
from src.feature_engineering import inverse_log_transform, log_transform_target
from src.pso_optimizer import (
    build_cnn_pso_objective,
    decode_cnn_hyperparameters,
    get_cnn_pso_bounds,
    tune_cnn_with_pso,
)

# 1) PSO tuning on TRAIN -> score on VALIDATION
objective_fn = build_cnn_pso_objective(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    input_length=X_train.shape[1],
    epochs=5,
    use_log_transform=True,
    verbose=0,
)

lower_bounds, upper_bounds = get_cnn_pso_bounds()
best_cost, best_position = tune_cnn_with_pso(
    objective_fn=objective_fn,
    dimensions=6,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    n_particles=6,
    iters=6,
)
best_params = decode_cnn_hyperparameters(best_position)

print("Best validation RMSE from PSO:", float(best_cost))
print("Best hyperparameters:", best_params)

# 2) Final training on TRAIN + VALIDATION (10 epochs)
X_train_final = np.vstack([X_train, X_val])
y_train_final = np.concatenate([y_train, y_val])

final_scaler = StandardScaler()
X_train_final_scaled = final_scaler.fit_transform(X_train_final).astype(np.float32)
X_test_final_scaled = final_scaler.transform(X_test).astype(np.float32)

y_train_final_fit = log_transform_target(y_train_final, use_log_transform=True).astype(np.float32)

X_train_final_cnn = reshape_for_cnn(X_train_final_scaled)
X_test_final_cnn = reshape_for_cnn(X_test_final_scaled)

final_model = build_cnn_regressor(
    input_length=X_train_final_cnn.shape[1],
    filters=int(best_params["filters"]),
    kernel_size=int(best_params["kernel_size"]),
    dense_units=int(best_params.get("dense_units", 16)),
    learning_rate=float(best_params["learning_rate"]),
    dropout_rate=float(best_params.get("dropout_rate", 0.2)),
    num_conv_layers=int(best_params.get("num_conv_layers", 2)),
    batch_size=int(best_params.get("batch_size", 16)),
)

final_model, final_history = train_cnn_model(
    model=final_model,
    x_train=X_train_final_cnn,
    y_train=y_train_final_fit,
    epochs=10,
    batch_size=int(best_params.get("batch_size", 16)),
    verbose=0,
    use_callbacks=False,
    validation_split=0.0,
)

# 3) Final evaluation on TEST
test_preds_log = final_model.predict(X_test_final_cnn, verbose=0).ravel()
test_preds = inverse_log_transform(test_preds_log, use_log_transform=True)

test_rmse = float(np.sqrt(mean_squared_error(y_test, test_preds)))
test_mae = float(mean_absolute_error(y_test, test_preds))

print("\nFinal TEST metrics")
print("RMSE:", test_rmse)
print("MAE:", test_mae)

sample_prediction = float(test_preds[0])
print(f"\nModel output = {sample_prediction:.2f}")
print(f"-> Predicted normalized effort value for the selected project = {sample_prediction:.2f}")
print("The model predicts normalized effort values, which can be inverse-transformed to actual effort.")

2026-04-29 16:03:27,712 - pyswarms.single.global_best - INFO - Optimize for 6 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|6/6, best_cost=0.639
2026-04-29 16:04:21,718 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.6392263206214015, best pos: [24.09228279  3.82447541 -2.03197572  0.30609424  2.72685104 11.75826349]


Best validation RMSE from PSO: 0.6392263206214015
Best hyperparameters: {'filters': 24, 'kernel_size': 4, 'dense_units': 64, 'learning_rate': 0.009290183207085701, 'dropout_rate': 0.3060942398917824, 'num_conv_layers': 3, 'batch_size': 8, 'learning_rate_exp': -2.0319757214162806}

Final TEST metrics
RMSE: 0.6902240729885462
MAE: 0.48763418616435467

Model output = 1.24
-> Predicted normalized effort value for the selected project = 1.24
The model predicts normalized effort values, which can be inverse-transformed to actual effort.
